In [1]:
import scanpy as sc
import pandas as pd
import os
from scipy.io import mmread
from scipy import sparse
import numpy as np
import anndata

In [2]:
def make_block_matrix(a, n, block_value=1, other_value=2):
    """
    Return an (a x a*n) matrix where for row i (0-based)
    columns [i*n, i*n + n - 1] are set to block_value,
    all other entries set to other_value.
    """
    b = a * n
    M = np.full((a, b), other_value, dtype=int)
    for i in range(a):
        start = i * n
        M[i, start:start + n] = block_value
    return M

In [31]:
def pseudobulk_from_anndata(adata,
                            groupby,            # e.g. "cell_type" (one value per cell)
                            n_pseudobulks=5,    # per group if using random-subsets
                            cells_per_pb=200,   # how many single cells to sum per pseudobulk
                            suffix = True,
                            random_seed=0):
    """
    Return: pandas DataFrame with index = genes (adata.var_names), columns = pseudobulk samples.
    Values are raw counts sums (numeric).
    """
    np.random.seed(random_seed)
    X = adata.X
    genes = adata.var_names.tolist()

    # ensure obs has groupby
    if groupby not in adata.obs.columns:
        raise KeyError(f"{groupby} not in adata.obs")

    pb_cols = {}
    # iterate groups
    groups = adata.obs[groupby].unique().tolist()
    for g in groups:
        mask = (adata.obs[groupby] == g).values
        idx = np.nonzero(mask)[0]
        if len(idx) == 0:
            continue

        cell_indices = idx.tolist()
        for i in range(n_pseudobulks):
            replace = len(cell_indices) < cells_per_pb
            chosen = (np.random.choice(cell_indices, size=cells_per_pb, replace=replace)
                        if cells_per_pb > 0 else np.array(cell_indices))
            if sparse.issparse(X):
                vec = X[chosen, :].sum(axis=0)
                vec = np.asarray(vec).ravel()
            else:
                vec = X[chosen, :].sum(axis=0)
                vec = np.asarray(vec).ravel()
            colname = f"{g}__pb{i+1}"
            pb_cols[colname] = vec

    
    class_mat = make_block_matrix(len(groups), n_pseudobulks)
    class_df = pd.DataFrame(class_mat,
                            index=groups)

    # make DataFrame: rows = genes, cols = pseudobulks
    if len(pb_cols) == 0:
        raise RuntimeError("No pseudobulks produced. Check groupby column and adata.obs.")
    pb_df = pd.DataFrame(pb_cols, index=genes)
    # ensure numeric dtype
    pb_df = pb_df.fillna(0).astype(float)
    if not suffix:
        pb_df.columns = [i.split('__')[0] for i in pb_df.columns]
    return pb_df,class_df

### Gut

In [4]:
adata_gut = sc.read_h5ad(r"E:\AAA_Labwork\Tcell tissues\v2\gut_annotated.h5ad") #that's the scRNAseq data we generated

In [5]:
adata_gut

AnnData object with n_obs × n_vars = 14218 × 29051
    obs: 'n_counts', 'log_counts', 'n_genes', 'log10GenesPerUMI', 'mt_frac', 'ribo_frac', 'batch', 'S_score', 'G2M_score', 'S_G2M_diff', 'phase', 'leiden', 'leiden2', 'leiden3', 'cluster_lowres', 'celltype_lowres', 'celltype_highres', 'Donor ID'
    var: 'highly_variable'
    uns: 'batch_colors', 'celltype_highres_colors', 'celltype_lowres_colors', 'leiden', 'leiden2_colors', 'leiden3_colors', 'leiden_colors', 'leiden_sizes', 'neighbors', 'paga', 'pca', 'rank', 'rank2', 'rankendo', 'ranksmooth', 'residuals', 'scaled', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts', 'logcounts', 'raw'
    obsp: 'connectivities', 'distances'

In [6]:
adata_gut.obs['celltype_lowres'].values.categories

Index(['T Cells', 'NK', 'ILC', 'B Cells', 'Plasma Cells', 'Monocytes',
       'Macrophages', 'Dendritic Cells', 'Endothelial Cells', 'Telocytes',
       'Fibroblastic Reticular Cells', 'Fibroblast', 'Smooth Muscle Cells',
       'Intestinal Epithelial Cells', 'Enteric Glial Cells'],
      dtype='object')

In [7]:
adata_epi_nonimmune = adata_gut[adata_gut.obs['celltype_lowres'] == 'Intestinal Epithelial Cells']

In [8]:
adata_epi_nonimmune.obs['celltype_highres']

AAACCTGTCAGCACAT-1-3                  Tuft Cells
AAAGATGTCGGTTAAC-1-3                 Colonocytes
AAAGCAACAGCTTCGG-1-3    Transit Amplifying Cells
AAAGTAGTCTGGTGTA-1-3    Transit Amplifying Cells
AAATGCCTCACATGCA-1-3                 Colonocytes
                                  ...           
TTGGCAAGTCCGTTAA-1-5                 Colonocytes
TTTATGCGTACACCGC-1-5                 Colonocytes
TTTATGCGTTCCTCCA-1-5                 Colonocytes
TTTGCGCTCTTGAGGT-1-5                 SPIB+ Cells
TTTGGTTAGCCAGGAT-1-5    Transit Amplifying Cells
Name: celltype_highres, Length: 1356, dtype: category
Categories (7, object): ['Colonocytes', 'Enteroendocrine Cells', 'Goblet Cells', 'Intestinal Stem Cells', 'SPIB+ Cells', 'Transit Amplifying Cells', 'Tuft Cells']

In [9]:
adata_epi_nonimmune_3 = adata_epi_nonimmune[adata_epi_nonimmune.obs['batch']=='3']
adata_epi_nonimmune_4 = adata_epi_nonimmune[adata_epi_nonimmune.obs['batch']=='4']


In [10]:
adata_lp_immune = adata_gut[adata_gut.obs['celltype_lowres'].isin(['T Cells', 'NK', 'ILC', 'B Cells', 'Plasma Cells', 'Monocytes',
       'Macrophages', 'Dendritic Cells'])]

In [11]:
adata_lp_immune_3 = adata_lp_immune[adata_lp_immune.obs['batch']=='3']
adata_lp_immune_4 = adata_lp_immune[adata_lp_immune.obs['batch']=='4']

In [22]:
adata_list = [adata_epi_nonimmune_3, adata_epi_nonimmune_4, adata_lp_immune_3, adata_lp_immune_4]
adata_names = ['adata_epi_nonimmune_3', 'adata_epi_nonimmune_4', 'adata_lp_immune_3', 'adata_lp_immune_4']
for i in range(len(adata_list)):
    adata = adata_list[i]
    name = adata_names[i]
    counts = adata.layers['raw']
    try:
        mat = counts.T.todense()   # works for sparse matrices
    except:
        mat = np.asarray(counts).T             # works for dense matrices
    if i > 1:
        df = pd.DataFrame(data=mat, 
                        index=adata.var_names, 
                        columns=adata.obs['celltype_highres'])
    else:
        df = pd.DataFrame(data=mat, 
                        index=adata.var_names, 
                        columns=adata.obs['celltype_lowres'])    
    df.index.name = 'GeneSymbol'
    df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\{name}.txt',sep='\t')

In [32]:
# try pseudobulk
adata_list = [adata_epi_nonimmune_3, adata_epi_nonimmune_4, adata_lp_immune_3, adata_lp_immune_4]
adata_names = ['adata_epi_nonimmune_3', 'adata_epi_nonimmune_4', 'adata_lp_immune_3', 'adata_lp_immune_4']
for i in range(len(adata_list)):
    adata = adata_list[i]
    name = adata_names[i]
    adata.X = adata.layers['raw']
    if i > 1:
        key = 'celltype_highres'
    else:
        key =  'celltype_lowres'
    pb, class_df = pseudobulk_from_anndata(adata, groupby=key, n_pseudobulks=10, cells_per_pb=100, suffix= False, random_seed=0)
    pb.index.name = 'GeneSymbol'
    pb.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\{name}_pb.txt',sep='\t')
    class_df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\{name}_class_df.txt',sep='\t',header = False)

c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
C:\Users\16220\AppData\Local\Temp\ipykernel_1488\406201200.py:7: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata.X = adata.layers['raw']
c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  war

I must say the experiment design is messed up--wish I could have a scRNAseq for IEL + epithelial, but I only have epithelial + IEL + LP CD45+ + others. I will concatenate the epithelials with the IEL from our previous study.

In [13]:
previous_adata = sc.read_h5ad(r"E:\AAA_Labwork\T cells\v3\250307_gut_liver_blood_ultimate_annotated.h5ad") #that's the scRNAseq data we generated

In [14]:
iel = previous_adata[previous_adata.obs['tissue']=='IEL']

In [15]:
iel = iel[iel.obs['major celltype']!='Other']

In [16]:
iel_3 = iel[iel.obs['batch']=='3']
iel_4 = iel[iel.obs['batch']=='4']

In [17]:
iel_3.X = iel_3.layers['counts']
iel_4.X = iel_4.layers['counts']

c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
C:\Users\16220\AppData\Local\Temp\ipykernel_1488\1329521237.py:1: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  iel_3.X = iel_3.layers['counts']
C:\Users\16220\AppData\Local\Temp\ipykernel_1488\1329521237.py:2: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  iel_4.X = iel_4.layers['counts']


In [18]:
adata_epi_nonimmune_3.obs['major celltype'] = adata_epi_nonimmune_3.obs['celltype_highres']
adata_epi_nonimmune_4.obs['major celltype'] = adata_epi_nonimmune_4.obs['celltype_highres']

C:\Users\16220\AppData\Local\Temp\ipykernel_1488\3142887664.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_epi_nonimmune_3.obs['major celltype'] = adata_epi_nonimmune_3.obs['celltype_highres']
C:\Users\16220\AppData\Local\Temp\ipykernel_1488\3142887664.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_epi_nonimmune_4.obs['major celltype'] = adata_epi_nonimmune_4.obs['celltype_highres']


In [19]:
mix_3 = anndata.AnnData.concatenate(iel_3, adata_epi_nonimmune_3, join="outer", batch_key="dataset")
mix_4 = anndata.AnnData.concatenate(iel_4, adata_epi_nonimmune_4, join="outer", batch_key="dataset")

C:\Users\16220\AppData\Local\Temp\ipykernel_1488\3023423861.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  mix_3 = anndata.AnnData.concatenate(iel_3, adata_epi_nonimmune_3, join="outer", batch_key="dataset")
C:\Users\16220\AppData\Local\Temp\ipykernel_1488\3023423861.py:2: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  mix_4 = anndata.AnnData.concatenate(iel_4, adata_epi_nonimmune_4, join="outer", batch_key="dataset")


In [20]:
del mix_3.layers
del mix_3.obsm
del mix_3.var
del mix_4.layers
del mix_4.obsm
del mix_4.var

In [33]:
iel_3_pb, iel_3_class_df = pseudobulk_from_anndata(iel_3, groupby='major celltype', n_pseudobulks=10, cells_per_pb=100, suffix=False, random_seed=0)
iel_3_pb.index.name = 'GeneSymbol'
iel_3_pb.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\iel_3_pb.txt',sep='\t')
iel_3_class_df.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\iel_3_class_df.txt',sep='\t',header = False)

In [34]:
iel_4_pb, iel_4_class_df = pseudobulk_from_anndata(iel_4, groupby='major celltype', n_pseudobulks=10, cells_per_pb=100, suffix=False, random_seed=0)
iel_4_pb.index.name = 'GeneSymbol'
iel_4_pb.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\iel_4_pb.txt',sep='\t')
iel_4_class_df.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\iel_4_class_df.txt',sep='\t',header = False)

In [35]:
mix_3_pb, mix_3_class_df = pseudobulk_from_anndata(mix_3, groupby='major celltype', n_pseudobulks=10, cells_per_pb=100, suffix=False, random_seed=0)
mix_3_pb.index.name = 'GeneSymbol'
mix_3_pb.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\mix_3_pb.txt',sep='\t')
mix_3_class_df.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\mix_3_class_df.txt',sep='\t',header = False)

In [36]:
mix_4_pb, mix_4_class_df = pseudobulk_from_anndata(mix_4, groupby='major celltype', n_pseudobulks=10, cells_per_pb=100, suffix=False, random_seed=0)
mix_4_pb.index.name = 'GeneSymbol'
mix_4_pb.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\mix_4_pb.txt',sep='\t')
mix_4_class_df.to_csv('C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\mix_4_class_df.txt',sep='\t',header = False)

### Liver

In [25]:
adata_liver = sc.read_h5ad(r"E:\AAA_Labwork\Tcell tissues\v2\liver_annotated.h5ad") #that's the scRNAseq data we generated
adata_liver

AnnData object with n_obs × n_vars = 22946 × 28283
    obs: 'n_counts', 'log_counts', 'n_genes', 'log10GenesPerUMI', 'mt_frac', 'ribo_frac', 'batch', 'S_score', 'G2M_score', 'S_G2M_diff', 'phase', 'leiden', 'leiden2', 'leiden3', 'celltype_lowres', 'celltype_highres', 'Donor ID'
    var: 'highly_variable'
    uns: 'Donor ID_colors', 'HVG', 'batch_colors', 'celltype_highres_colors', 'celltype_lowres_colors', 'leiden', 'leiden2_colors', 'leiden3_colors', 'leiden_colors', 'leiden_sizes', 'neighbors', 'paga', 'pca', 'rank', 'residuals', 'scaled', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts', 'logcounts', 'raw'
    obsp: 'connectivities', 'distances'

In [26]:
adata_liver.obs['celltype_lowres'].values.categories

Index(['T Cells', 'NK', 'ILC', 'B Cells', 'Plasma Cells', 'Macrophages',
       'Dendritic Cells', 'Erthyroid', 'Mast Progenitor', 'PEC', 'LSEC',
       'Stellate Cells', 'Cholangiocytes', 'Hepatocytes', 'Glial-like Cells'],
      dtype='object')

In [27]:
adata_liver_immune = adata_liver[adata_liver.obs['celltype_lowres'].isin(['T Cells', 'NK', 'ILC', 'B Cells', 'Plasma Cells', 'Monocytes',
       'Macrophages', 'Dendritic Cells'])]
adata_liver_immune_3 = adata_liver_immune[adata_liver_immune.obs['batch']=='3']
adata_liver_immune_4 = adata_liver_immune[adata_liver_immune.obs['batch']=='4']

In [ ]:
adata_list = [adata_liver_immune_3, adata_liver_immune_4]
adata_names = ['adata_liver_immune_3', 'adata_liver_immune_4']
for i in range(len(adata_list)):
    adata = adata_list[i]
    name = adata_names[i]
    subsampled = sc.pp.subsample(adata, n_obs=5000, random_state=0,copy=True)
    counts = subsampled.layers['raw']
    try:
        mat = counts.T.todense()   # works for sparse matrices
    except:
        mat = np.asarray(counts).T             # works for dense matrices
    if i > 1:
        df = pd.DataFrame(data=mat, 
                        index=subsampled.var_names, 
                        columns=subsampled.obs['celltype_highres'])
    else:
        df = pd.DataFrame(data=mat, 
                        index=subsampled.var_names, 
                        columns=subsampled.obs['celltype_lowres'])    
    df.index.name = 'GeneSymbol'
    df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\{name}.txt',sep='\t')

In [37]:
adata_list = [adata_liver_immune_3, adata_liver_immune_4]
adata_names = ['adata_liver_immune_3', 'adata_liver_immune_4']
for i in range(len(adata_list)):
    adata = adata_list[i]
    name = adata_names[i]
    adata.X = adata.layers['raw']
    if i > 1:
        key = 'celltype_highres'
    else:
        key =  'celltype_lowres'
    pb, class_df = pseudobulk_from_anndata(adata, groupby=key, n_pseudobulks=10, cells_per_pb=100, suffix= False, random_seed=0)
    pb.index.name = 'GeneSymbol'
    pb.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\{name}_pb.txt',sep='\t')
    class_df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\by_donor\\pb\\{name}_class_df.txt',sep='\t',header = False)

c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
C:\Users\16220\AppData\Local\Temp\ipykernel_1488\1941056278.py:6: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata.X = adata.layers['raw']
c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  wa